In [ ]:
#Get Smart Contract Addresses from EthereumETL
import numpy as np
import pandas as pd
import os
import time
import zipfile
from tqdm import tqdm
from google.cloud import bigquery
os.environ["GOOGLE_APPLICATION_CREDENTIALS"]="application_default_credentials.json" #Replace it with your own GOOGLE_APPLICATION_CREDENTIALS (BigQuery)
client = bigquery.Client(project="sc-mde-nymmpx") #Replace the project name to your project name

In [ ]:
# Construct a reference to the "Ethereum Blockchain" dataset
eth_ref = client.dataset("crypto_ethereum", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(eth_ref)

table_ref = eth_ref.table("contracts")
transaction_table = client.get_table(table_ref)
query = """
WITH contracts_in_block as ( 
    SELECT
      address,
      bytecode,
      block_timestamp
    FROM
      `bigquery-public-data.crypto_ethereum.contracts`
)

SELECT
    *
FROM
    contracts_in_block
ORDER BY
    1 asc
"""

In [ ]:
query_job = client.query(query) 

In [ ]:
addresses = query_job.to_dataframe()
addresses.describe()

In [ ]:
#get Source codes from Etherscan for smart contract versions
import json
import os
import requests
import time
import pandas as pd

# Read the dataframe from a file or create it manually
df = pd.read_csv('smartContractAddresses.csv')  # Replace 'your_file.csv' with your actual file that has addresses of the contract

# Assuming the first column is named 'Header'
header_column = df['Addresses']
df.describe()
ETHERSCAN_API_KEY = "" #Use your own Etherscan API
BASE_DIR = "/path" #Replace with the path where you want to save the smart contracts source codes 


In [ ]:
def retrieve_source_codes(address: str):
    if len(address) != 42:
        return None

    url = f"https://api.etherscan.io/api?module=contract&action=getsourcecode&address={address}&apikey={ETHERSCAN_API_KEY}"
    try:
        response = requests.get(url)
        data = response.json()
        if data["status"] != "1":
            return None
        
        info = data["result"][0]
        sourcecode = info['SourceCode']
        contract_name = info['ContractName']
        files = []
        try:
            contract_object = json.loads(sourcecode[1:-1])
            files = [(name, code['content']) for name, code in contract_object.get('sources', {}).items()]
            # the file at index 0 is the root
            root = files[0][0]
        except json.JSONDecodeError:
            lines = sourcecode.splitlines()
            indices = [i for i, line in enumerate(lines) if line.startswith("// Dependency file:") or line.startswith("// Root file:")]
            files = [(lines[start].split(":")[1].strip(), '\n'.join(lines[start+1:end])) for start, end in zip(indices, indices[1:] + [len(lines)])]
            if len(files) == 0:
                files = [(f"{contract_name}.sol", sourcecode)]
                root = contract_name
            # the last file should be the root file
            root = files[-1][0]
        return (root.replace(" ", "_").replace("\\", "/").replace(".sol", "")).split("/")[-1], contract_name, files
    except requests.RequestException as e:
        print(f"Error: Failed to retrieve source code for address {address}. Exception: {e}")
        return None

In [ ]:
for address in header_column:
    try:
        root, name, sources = retrieve_source_codes(address)
        if not sources:
            continue

        for filename, code in sources:
            filename = filename.replace(" ", "_").replace("\\", "/")
            # here we save the files
            directory = f"{BASE_DIR}/{address}-{name}/{'/'.join(filename.split('/')[:-1])}"
            if not os.path.exists(directory):
                os.makedirs(directory)
            with open(f'{BASE_DIR}/{address}-{name}/{filename}', 'w') as f:
                f.write(code)

        # Append the address and root to a different file to track found contracts
        with open("found_contracts.txt", "a") as f:
            f.write(f"{address},{root}\n")

        print(f"{address},{root}")
    except Exception as e:
        print(e)